# LLM judge check

Interactively test the LLM-as-judge stage (`langtrend/judge.py`) on individual papers from a processed week.

Run from within `notebooks/` (paths are relative, e.g. `../data/processed/...`).
Configuration comes from `../.env` — see `.env.example` at the repo root. The default backend is
Groq's free tier; point `LLM_JUDGE_BASE_URL` at a local Ollama server for quota-free testing.

In [ ]:
import sys, json
from pathlib import Path

sys.path.insert(0, "..")

from langtrend.judge import (
    assemble_context, build_messages, collect_target_languages,
    judge_paper, safe_paper_id, save_judge_record,
)
from langtrend.llm_client import LLMClientConfig, OpenAICompatClient

config = LLMClientConfig.from_env()
client = OpenAICompatClient(config)
client.ping()  # raises with an actionable message if the endpoint/key is bad
print(f"Judge: {config.model} @ {config.base_url} (rpm={config.rpm}, context<={config.max_context_chars} chars)")

## Pick a week and load its detected papers

In [ ]:
WEEK = "20260518_to_20260525"  # <- change me

week_dir = Path(f"../data/processed/weeks/{WEEK}")
detected_path = week_dir / f"arxiv_papers_{WEEK}_detected.jsonl"

records = []
with detected_path.open(encoding="utf-8") as fh:
    for line in fh:
        if line.strip():
            records.append(json.loads(line))
print(f"{len(records)} detected papers in {WEEK}")

## List papers with low-resource (class 0–4) detections

These are the interesting candidates — class-5-only papers (English/Chinese etc.) rarely need review.

In [ ]:
for record in records[:40]:  # bump the slice to see more
    targets = collect_target_languages(record, classes={0, 1, 2, 3, 4})
    if not targets:
        continue
    langs = ", ".join(f"{t['language']}({t['class']})" for t in targets)
    print(f"{safe_paper_id(record['paper_id']):<16} {record['paper']['title'][:70]:<72} {langs}")

## Inspect one paper: assembled context + prompt

In [ ]:
PAPER_ID = "2605.17710v1"  # <- change me (bare safe_id from the list above)

record = next(r for r in records if safe_paper_id(r["paper_id"]) == PAPER_ID)
targets = collect_target_languages(record)
context = assemble_context(record, week_dir, targets, max_chars=config.max_context_chars)
messages = build_messages(context, targets)

print(record["paper"]["title"])
print(f"targets: {[t['language'] for t in targets]}")
print(f"context: {context.total_chars} chars, coverage={context.coverage}, {len(context.snippets)} snippets")
print()
print(messages[1]["content"][:4000])  # user prompt preview

## Judge the paper and compare against the regex detections

In [ ]:
judge_record = judge_paper(record, week_dir, client, config)

width = max(len(name) for name in judge_record["verdicts"]) if judge_record["verdicts"] else 10
print(f"model: {judge_record['judge_model']}  coverage: {judge_record['context_coverage']}")
print()
for target in targets:
    v = judge_record["verdicts"].get(target["language"])
    verdict = v["verdict"] if v else "(unjudged)"
    reason = v["reason"] if v else ""
    print(f"{target['language']:<{width}}  class {target['class']}  {verdict:<15} {reason}")

## Compare with an existing cached verdict (if `make judge` already ran)

In [ ]:
cache_path = week_dir / "judge_cache" / f"{PAPER_ID}.json"
if cache_path.exists():
    cached = json.loads(cache_path.read_text(encoding="utf-8"))
    print(f"cached at {cached['judged_at']} by {cached['judge_model']}")
    for lang, v in cached["verdicts"].items():
        fresh = judge_record["verdicts"].get(lang, {}).get("verdict", "(unjudged)")
        agree = "==" if fresh == v["verdict"] else "!="
        print(f"{lang:<20} cached={v['verdict']:<15} {agree} fresh={fresh}")
else:
    print(f"No cached verdict yet: {cache_path}")
    # save_judge_record(week_dir, judge_record)  # uncomment to write the cache

## Optional: judge a small sample and tabulate the verdict distribution

In [ ]:
from collections import Counter

SAMPLE = 5  # papers to judge (each = 1 API call; mind the free-tier quota)

distribution = Counter()
for record in records:
    targets = collect_target_languages(record, classes={0, 1, 2, 3, 4})
    if not targets:
        continue
    jr = judge_paper(record, week_dir, client, config)
    for lang, v in jr["verdicts"].items():
        distribution[v["verdict"]] += 1
    SAMPLE -= 1
    if SAMPLE <= 0:
        break

print(dict(distribution))